In [1]:
import pandas as pd
import re
from pathlib import Path

## Cleaning of PropertyPro.ng Dataset

In [2]:
BASE_DIR = Path.cwd().parent  
RAW = BASE_DIR / "Data" / "raw"
df_original = pd.read_csv(RAW / "ppro_abuja_rentals.csv")

In [3]:
df_original.shape

(2177, 4)

In [4]:
df_original.head()

,Property Category,Price (Per Annum),Location,Scraped_At
0,Newly Built Furnished&serviced 2&3 Bed Block O...,"$ 45,000",Maitama By Fruit Market Maitama Abuja,2026-06-01 11:33:21
1,4 Bedroom Cornerpiece Terrace Duplex Mabushi,"₦ 20,000,000/year",Mabushi Abuja,2026-06-01 11:33:21
2,2 Bedroom Apartment,"₦ 8,500,000/year",Nnpc Estate Life Camp Abuja,2026-06-01 11:33:21
3,Beautiful And Well Finished 4bedroom Terrace W...,"₦ 20,000,000",Katampe Extention Diplomatic Zone Katampe Exte...,2026-06-01 11:33:21
4,6 Bedroom Detached Duplex With Bq,"₦ 45,000,000/year",Near Amigo Wuse 2 Abuja,2026-06-01 11:33:21


In [5]:
df = df_original.copy()
df.head()

,Property Category,Price (Per Annum),Location,Scraped_At
0,Newly Built Furnished&serviced 2&3 Bed Block O...,"$ 45,000",Maitama By Fruit Market Maitama Abuja,2026-06-01 11:33:21
1,4 Bedroom Cornerpiece Terrace Duplex Mabushi,"₦ 20,000,000/year",Mabushi Abuja,2026-06-01 11:33:21
2,2 Bedroom Apartment,"₦ 8,500,000/year",Nnpc Estate Life Camp Abuja,2026-06-01 11:33:21
3,Beautiful And Well Finished 4bedroom Terrace W...,"₦ 20,000,000",Katampe Extention Diplomatic Zone Katampe Exte...,2026-06-01 11:33:21
4,6 Bedroom Detached Duplex With Bq,"₦ 45,000,000/year",Near Amigo Wuse 2 Abuja,2026-06-01 11:33:21


In [6]:
# print all rows where the price contains a dollar sign
for price in df["Price (Per Annum)"]:
    if "$" in str(price):
        print(price)

$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 40,000/year
$ 15,000/year
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 65,000/year
$ 37,000/year
$ 16,000/year
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 150,000
$ 45,000
$ 45,000
$ 35,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 35,000
$ 45,000
$ 150,000
$ 150,000
$ 150,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000
$ 45,000


In [7]:
#drop rows with /day and /sqm from price() column
df = df[~df['Price (Per Annum)'].str.contains('/day|/sqm|/month', case=False, na=False)].copy()

In [8]:
df['Price (Per Annum)'] = (
    df['Price (Per Annum)']
    .astype(str)
    .str.replace('/year', '', case=False, regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)

In [9]:
exchange_rate = 1370.43
col = 'Price (Per Annum)'

df[col] = df[col].astype(str).str.strip()

#flag dollar rows
is_dollar = df[col].str.startswith('$', na=False)
#drop currency symbols
df[col] = df[col].str.replace('$', '', regex=False)
df[col] = df[col].str.replace('₦', '', regex=False)

#convert to float
df[col] = pd.to_numeric(df[col], errors='coerce')

#multiply only the dollar rows by the exchange rate
df.loc[is_dollar, col] = df.loc[is_dollar, col] * exchange_rate

df.head()

,Property Category,Price (Per Annum),Location,Scraped_At
0,Newly Built Furnished&serviced 2&3 Bed Block O...,61669350,Maitama By Fruit Market Maitama Abuja,2026-06-01 11:33:21
1,4 Bedroom Cornerpiece Terrace Duplex Mabushi,20000000,Mabushi Abuja,2026-06-01 11:33:21
2,2 Bedroom Apartment,8500000,Nnpc Estate Life Camp Abuja,2026-06-01 11:33:21
3,Beautiful And Well Finished 4bedroom Terrace W...,20000000,Katampe Extention Diplomatic Zone Katampe Exte...,2026-06-01 11:33:21
4,6 Bedroom Detached Duplex With Bq,45000000,Near Amigo Wuse 2 Abuja,2026-06-01 11:33:21


In [10]:
#create bedroom column from property category column
df['Bedrooms'] = df['Property Category'].str.extract(r'(\d+)\s*[Bb]ed', flags=re.IGNORECASE)

df['Bedrooms'] = pd.to_numeric(df['Bedrooms'], errors='coerce')
self_contain_mask = df['Property Category'].str.contains('Self', case=False, na=False)
df.loc[self_contain_mask, 'Bedrooms'] = 1
df[['Property Category', 'Bedrooms']].head()

,Property Category,Bedrooms
0,Newly Built Furnished&serviced 2&3 Bed Block O...,3.0
1,4 Bedroom Cornerpiece Terrace Duplex Mabushi,4.0
2,2 Bedroom Apartment,2.0
3,Beautiful And Well Finished 4bedroom Terrace W...,4.0
4,6 Bedroom Detached Duplex With Bq,6.0


In [11]:
df.head()

,Property Category,Price (Per Annum),Location,Scraped_At,Bedrooms
0,Newly Built Furnished&serviced 2&3 Bed Block O...,61669350,Maitama By Fruit Market Maitama Abuja,2026-06-01 11:33:21,3.0
1,4 Bedroom Cornerpiece Terrace Duplex Mabushi,20000000,Mabushi Abuja,2026-06-01 11:33:21,4.0
2,2 Bedroom Apartment,8500000,Nnpc Estate Life Camp Abuja,2026-06-01 11:33:21,2.0
3,Beautiful And Well Finished 4bedroom Terrace W...,20000000,Katampe Extention Diplomatic Zone Katampe Exte...,2026-06-01 11:33:21,4.0
4,6 Bedroom Detached Duplex With Bq,45000000,Near Amigo Wuse 2 Abuja,2026-06-01 11:33:21,6.0


In [12]:
import numpy as np
def find_property_type(title):
    if pd.isna(title):
        return np.nan
    
    t = title.lower()
    
    if 'semi detached duplex' in t or 'semi-detached' in t:
        return 'Semi-Detached Duplex'
    elif 'terraced duplex' in t or 'terrace duplex' in t or 'terrace' in t:
        return 'Terraced Duplex'
    elif 'detached duplex' in t:
        return 'Detached Duplex'
    elif 'duplex' in t or 'rooms duplex' in t:
        return 'Detached Duplex' 
    elif 'self contain' in t:
        return 'Self Contain'
    elif 'flat' in t or 'apartment' in t:
        return 'Apartment'
    elif 'bungalow' in t:
        return 'Detached Bungalow'
    
    return np.nan

df['Property Type'] = df['Property Category'].apply(find_property_type)

In [16]:
df = df.dropna(subset=['Property Type'])
category_map = {
    'Self Contain': 'Apartment',
    'Apartment': 'Apartment',
    'Detached Duplex': 'Duplex',
    'Semi-Detached Duplex': 'Duplex',
    'Terraced Duplex': 'Duplex',
    'Detached Bungalow': 'Bungalow'
}
df['Property Category'] = df['Property Type'].map(category_map)

df[['Property Category', 'Property Type']].drop_duplicates()

,Property Category,Property Type
0,Apartment,Apartment
1,Duplex,Terraced Duplex
4,Duplex,Detached Duplex
12,Duplex,Semi-Detached Duplex
21,Apartment,Self Contain
147,Bungalow,Detached Bungalow


In [17]:
#find null values
num_isnull = df.isnull().sum()
print(num_isnull)

Property Category    0
Price (Per Annum)    0
Location             0
Scraped_At           0
Bedrooms             0
Property Type        0
dtype: int64


In [22]:
amac_districts = ['Jabi', 'Kaura', 'Garki', 'Kabusa', 'City Centre', 'Wuse', 'Gwarinpa', 'Gui', 'Karshi', 'Asokoro', 'Jahi', 'Guzape', 'Apo', 'Durumi', 'Lugbe', 'Lokogoma', 'Maitama', 'Wuye', 'Katampe', 'Life Camp', 'Utako', 'Mabushi', 'Idu', 'Kado', 'Galadimawa', 'Karu', 'Gaduwa', 'Gudu', 'kukwaba', 'Karmo', 'Kubwa', 'Central Business District', 'CBD', 'Karsana']

district_pattern = '|'.join(amac_districts)

#extract the district name 
df['District'] = df['Location'].str.extract(f'({district_pattern})', flags=re.IGNORECASE, expand=False)

df['District'] = df['District'].str.title()
print(df['District'].value_counts(dropna=False))

District
Guzape        193
Jahi          166
Lugbe         150
Maitama       142
Life Camp     138
Katampe       120
Gwarinpa      118
Asokoro       102
Wuse           98
Kubwa          91
NaN            79
Mabushi        75
Wuye           60
Garki          47
Apo            42
Jabi           36
Lokogoma       35
Durumi         30
Galadimawa     26
Idu            25
Kado           20
Utako          14
Gaduwa         14
Kaura           9
Karsana         5
Kukwaba         3
Kabusa          3
Gudu            2
Karmo           2
Name: count, dtype: int64


In [19]:
new_order = [
    'District', 
    'Bedrooms', 
    'Price (Per Annum)', 
    'Property Category', 
    'Property Type', 
    'Location',   
    'Scraped_At'    
]

df = df[new_order]

df.head()

,District,Bedrooms,Price (Per Annum),Property Category,Property Type,Location,Scraped_At
0,Maitama,3.0,61669350,Apartment,Apartment,Maitama By Fruit Market Maitama Abuja,2026-06-01 11:33:21
1,Mabushi,4.0,20000000,Duplex,Terraced Duplex,Mabushi Abuja,2026-06-01 11:33:21
2,Life Camp,2.0,8500000,Apartment,Apartment,Nnpc Estate Life Camp Abuja,2026-06-01 11:33:21
3,Katampe,4.0,20000000,Duplex,Terraced Duplex,Katampe Extention Diplomatic Zone Katampe Exte...,2026-06-01 11:33:21
4,Wuse,6.0,45000000,Duplex,Detached Duplex,Near Amigo Wuse 2 Abuja,2026-06-01 11:33:21


In [25]:
df.dropna(subset=['District'], inplace=True)
num_isnull = df.isnull().sum()
print(num_isnull)

District             0
Bedrooms             0
Price (Per Annum)    0
Property Category    0
Property Type        0
Location             0
Scraped_At           0
dtype: int64


In [26]:
df.shape

(1766, 7)

In [ ]:
df.to_csv(BASE_DIR / "Data" / "processed" / "ppro_cleaned.csv", index=False)